rag- retrieval agumentation generation

1)Prepare the document
2)Answer question

1---
i. upload the document
ii.split into chunks
iii.embed
iv.store(vector store)
2---
i.embed user question
ii.find closest chunk
iii.build prompt to and generate the answer

In [2]:
!pip install -q -U google-genai pypdf

In [4]:
from google import genai
from google.genai import types
from google.colab import userdata
import numpy as np

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))


In [20]:
#1.Prepare the document
#1.1 Upload the document
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF UPLOADED!!\nPDF NAME:{pdf_name}")

Saving Aircraft_Engine_IoT_AI_Health_Monitoring_Project.pdf to Aircraft_Engine_IoT_AI_Health_Monitoring_Project (3).pdf
PDF UPLOADED!!
PDF NAME:Aircraft_Engine_IoT_AI_Health_Monitoring_Project (3).pdf


In [26]:
#1.2 Extract text from document
from pypdf import PdfReader
reader=PdfReader(pdf_name)
print("Number of pages in PDF: ",len(reader.pages))

#extracting text to variable
text=""
for page in reader.pages:
  text+=page.extract_text()+"\n"
print("TEXT EXTRACTED SUCEESSFULLY\n")
print("NUMBER OF CHARACTERS: ",len(text))
print(text)

Number of pages in PDF:  4
TEXT EXTRACTED SUCEESSFULLY

NUMBER OF CHARACTERS:  5369
IoT & AI-Based Aircraft Engine Health Monitoring
 and Early Fault Detection System
 Aeronautical Engineering Mini Project — Complete Base Concept
1. Basic Idea
This project develops a miniature aircraft engine health monitoring system. Temperature, vibration and pressure
sensors continuously collect engine-condition data. An ESP32 processes the readings and sends them over Wi-Fi to
an IoT dashboard. The system provides real-time monitoring, abnormal-condition alerts and an optional AI/ML feature
for predicting engine health. A DC motor or fan-based setup can be used instead of a real aircraft engine for
demonstration.
2. Problem Statement
Aircraft engines operate under high temperature, pressure and vibration. Changes in these parameters can indicate
abnormal operating conditions. A low-cost prototype is therefore required to continuously monitor important engine
parameters, provide real-time visibility

In [52]:
#1.3 Chunking (overlapping chunking)
def chunk_text(text,chunk_size=800,overlap=20):
  start=0
  chunks=[]
  while start<len(text):
    end=start+chunk_size
    chunks.append(text[start:end])
    start=end-overlap
  return chunks
chunks=chunk_text(text)
print("NUMBER OF CHUNKS: ",len(chunks))

NUMBER OF CHUNKS:  7


In [47]:
#1.4 Embed Every chunk
EMB_MODEL="gemini-embedding-001"
EMB_DIM=768
def embed_chunk(chunk):
  response=client.models.embed_content(
      model=EMB_MODEL,
      contents=chunk,
      config=types.EmbedContentConfig(
          output_dimensionality=EMB_DIM
      )
  )
  return response.embeddings[0].values

chunk_embeddings=[]
for i,chunk in enumerate(chunks):

  chunk_embeddings.append(embed_chunk(chunk))

print("EMBEDDING DONE\n")
chunk_embeddings=np.array(chunk_embeddings)
print(chunk_embeddings.shape)

EMBEDDING DONE

(7, 768)


In [50]:
#1.4. Finding best chunk using cosine similarity
def cosine_sim(a,b):
  a=np.array(a)
  b=np.array(b)

  return float((np.dot(a,b))/(np.linalg.norm(a)*(np.linalg.norm(b))))



In [51]:
def retrieve(question,k=3):

  question_embed=embed_chunk(question)

  similarity=[]
  for i,chunk_embed in enumerate(chunk_embeddings):
    score=cosine_sim(question_embed,chunk_embed)
    similarity.append((i,score))

  similarity.sort(key=lambda x:x[1],reverse=True)

  similarity=similarity[:k]
  print("FOUND NEAREST TOPIC RELAVENT TO QUESTION\n")

  return similarity

print(retrieve("SUMMERISE THIS PROJECT"))

FOUND NEAREST TOPIC RELAVENT TO QUESTION

[(6, 0.5694989525359736), (5, 0.5534252074491538), (4, 0.5515136899887233)]


In [54]:
model="gemini-3.5-flash-lite"
system_instruction=f""""You are a helpful PDF assistant that answers questions and summarizes information from the uploaded PDF .
                                Do not assume and provide any information if not provided in the pdf.
                                Do reply politely, if any bad/unethical text is given by user reply with politely dont use like this based on text
                                provide answer in structured way.
                                In the begining of session great   HI I AM HELPFULL PDF ASSISTANT, TELL THE INFORMATION YOU WANT FROM PDF
                          """
chats=client.chats.create(
    model=model,
    config=types.GenerateContentConfig(
        temperature=0.6,
        max_output_tokens=1500,
        system_instruction=system_instruction,
        thinking_config=types.ThinkingConfig(thinking_level='medium')
    ),
    history=[]
)
print("NOTE:Enter exit/quit/bye to exit session")
while True:
  user_input=input("AI:Enter the query: ")
  if user_input.lower() in ['exit','bye','quit']:
    print("EXITING.....\nEXITED THANKYOU.......")
    break

  similarity=retrieve(user_input)
  sim_chunks=[]
  for index,scores in similarity:
    sim_chunks.append(chunks[index])

  prompt=f"""
  USE THE CHUNKS FOUND SIMILAR={sim_chunks} TO QUESTION={user_input} to answer the questions.
  """
  response=chats.send_message(prompt)
  print(f"AI:{response.text}")


NOTE:Enter exit/quit/bye to exit session
AI:Enter the query: summerize pdf document
FOUND NEAREST TOPIC RELAVENT TO QUESTION

AI:HI I AM HELPFULL PDF ASSISTANT, TELL THE INFORMATION YOU WANT FROM PDF

Based on the provided document, here is a structured summary of the **IoT & AI-Based Aircraft Engine Health Monitoring and Early Fault Detection System**:

### 1. Project Overview & Basic Idea
* **Purpose:** A miniature aircraft engine health monitoring system (using a DC motor or fan-based setup for demonstration) designed for early fault detection.
* **Core Components:** Temperature, vibration, and pressure sensors continuously collect data, which is processed by an **ESP32** and sent over Wi-Fi to an IoT dashboard.
* **Problem Statement:** Aircraft engines operate under extreme conditions (high temperature, pressure, and vibration), and parameter changes indicate abnormal operating conditions.

### 2. Engine Health States
The system classifies conditions into three states:
* **NORMAL:*

In [30]:
for m in client.models.list():
  print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gem